# Polymer 4D-STEM peak detection with the pinned paper model

This workflow resolves an immutable, checksum-verified model, reuses its local cache, loads a calibrated 4D-STEM scan, detects peaks, builds count maps and flowlines, and exports selected figures. The model record remains private during review, so set `QUANTEM_POLYMER_MODEL_DIR` to the supplied local archive. Once the immutable public record exists, remove that override; the same pinned call will download and cache it atomically.

## Installation

Install the released QuantEM version containing this workflow. During PR review, an isolated environment may instead install the `paper/polymers` branch.

In [ ]:
import os
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import quantem as em
from quantem.diffraction import BraggPeaksPolymer, resolve_polymer_model

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
SCAN_PATH = Path(os.environ['POLYMER_4DSTEM_PATH'])
MODEL_DIR = os.environ.get('QUANTEM_POLYMER_MODEL_DIR')
OUTPUT_DIR = Path(os.environ.get('POLYMER_TUTORIAL_OUTPUT_DIR', 'polymer_tutorial_outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE, SCAN_PATH, OUTPUT_DIR

## Resolve the immutable model

Omitting both `version` and `latest` deliberately selects the paper-pinned version. Use `latest=True` only when reproducibility against the paper is not required.

In [ ]:
model = resolve_polymer_model(local_model_dir=MODEL_DIR)
cached = resolve_polymer_model(local_model_dir=MODEL_DIR)
assert cached.weights_path == model.weights_path
print(model.model_id, model.version, model.checksum)
print(model.weights_path)

## Load and calibrate the scan

The paper scan is a DigitalMicrograph file whose 4D signal is dataset 1. Detector binning is applied before inference. For another instrument, change the reader arguments and confirm the reciprocal or angular calibration metadata before interpreting radial positions.

In [ ]:
scan = em.io.read_4dstem(SCAN_PATH, file_type='digitalmicrograph', dataset_index=1)
max_scan = int(os.environ.get('POLYMER_TUTORIAL_MAX_SCAN', '0'))
if max_scan > 0:
    scan = scan[:max_scan, :max_scan]
scan = scan.bin(bin_factors=(1, 1, 4, 4))
print('shape:', scan.shape, 'sampling:', scan.sampling, 'units:', scan.units)

## Pinned-model inference

The experimental normalization uses scan-level percentiles 1 and 99, as recorded in the model specification. A circular sample mask keeps normalization and BatchNorm adaptation away from scan corners.

In [ ]:
def compute_parameters(data, lower_percentile=1.0, upper_percentile=99.0):
    if isinstance(data, torch.Tensor):
        values = data.detach().float().flatten()
        return tuple(torch.quantile(values, torch.tensor([lower_percentile, upper_percentile], device=values.device) / 100).cpu().tolist())
    return tuple(np.percentile(np.asarray(data), [lower_percentile, upper_percentile]))

def normalize_data(data, lower, upper):
    if isinstance(data, torch.Tensor):
        return torch.clamp(data, lower, upper).sub(lower).div(upper - lower + 1e-8)
    if upper == lower:
        return np.zeros_like(data, dtype=np.float32)
    return (np.clip(data, lower, upper) - lower) / (upper - lower)

yy, xx = np.ogrid[:scan.shape[0], :scan.shape[1]]
cy, cx = (np.array(scan.shape[:2]) - 1) / 2
sample_mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= (0.47 * min(scan.shape[:2])) ** 2

In [ ]:
bp = BraggPeaksPolymer.from_data(
    scan,
    device=DEVICE,
    compute_parameters=partial(compute_parameters, lower_percentile=1.0, upper_percentile=99.0),
    normalize_data=normalize_data,
)
bp.set_model_weights(
    model_id=model.model_id, version=model.version, local_model_dir=MODEL_DIR
)
bp.find_peaks_model(
    device=DEVICE, scan_mask=sample_mask, threshold_peak=0.5,
    n_normalize_samples=1000, initial_chunk_size=100, accelerating_voltage_kv=300,
)
bp.process_polar(scan_mask=sample_mask, center_device=DEVICE)

## Count maps, flowlines, and selected exports

The radial ranges below are examples in inverse angstroms; choose scientifically justified windows from the radial peak profile for a different specimen. `orientation_offset_degrees=-6.8` is the paper scan's calibrated display rotation.

In [ ]:
q_ranges = np.array([[0.00, 0.10], [0.10, 0.20], [0.20, 0.30]])
fig, axes, count_maps = bp.plot_peak_count_map(q_ranges, return_values=True)
fig.savefig(OUTPUT_DIR / 'polymer_peak_count_maps.png', dpi=180, bbox_inches='tight')
plt.close(fig)

In [ ]:
orientation = bp.make_orientation_histogram(
    radial_ranges=q_ranges, orientation_offset_degrees=-6.8, upsample_factor=1,
    theta_step_deg=2, progress_bar=True,
)
flowlines = bp.make_flowline_map(orientation, progress_bar=True)
flowline_rgb = bp.make_flowline_rainbow_image(
    flowlines, sum_radial_bins=True, plot_images=False, white_background=True
)
plt.imsave(OUTPUT_DIR / 'polymer_flowlines.png', np.asarray(flowline_rgb))

In [ ]:
bp.save_peak_figures(
    ry=scan.shape[0] // 2, rx=scan.shape[1] // 2,
    prefix='paper_model', save_dir=OUTPUT_DIR, show_polar=True,
)
sorted(path.name for path in OUTPUT_DIR.iterdir())